<a href="https://colab.research.google.com/github/lukwac123/machine_learning_bootcamp/blob/main/unsupervised/)4_anomaly_detection/03_anomaly_detection_time_series.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import prophet

np.random.seed(41)
prophet.__version__

'1.3.0'

In [3]:
df = pd.read_csv('https://storage.googleapis.com/esmartdata-courses-files/ml-course/traffic.csv',
                 parse_dates=['timestamp'])
df.head()

,timestamp,count
0,2018-09-25 14:01:00,182.478
1,2018-09-25 14:02:00,176.231
2,2018-09-25 14:03:00,183.917
3,2018-09-25 14:04:00,177.798
4,2018-09-25 14:05:00,165.469


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14398 entries, 0 to 14397
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  14398 non-null  datetime64[ns]
 1   count      14398 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 225.1 KB


In [5]:
px.line(df, x='timestamp', y='count', title='Anomaly Detection - web traffic', width=950, height=500,
        template='plotly_dark', color_discrete_sequence=['#42f5d4'])

In [6]:
px.scatter(df, x='timestamp', y='count', title='Anomaly Detection - web traffic', width=950, height=500,
           template='plotly_dark', color_discrete_sequence=['#42f5d4'])

In [7]:
df.head()

,timestamp,count
0,2018-09-25 14:01:00,182.478
1,2018-09-25 14:02:00,176.231
2,2018-09-25 14:03:00,183.917
3,2018-09-25 14:04:00,177.798
4,2018-09-25 14:05:00,165.469


In [8]:
data = df.copy()
data.columns = ['ds', 'y']
data.head(3)

,ds,y
0,2018-09-25 14:01:00,182.478
1,2018-09-25 14:02:00,176.231
2,2018-09-25 14:03:00,183.917


In [9]:
from prophet import Prophet

Prophet?

In [10]:
model = Prophet(daily_seasonality=True, yearly_seasonality=False, weekly_seasonality=False,
                interval_width=0.99, changepoint_range=0.8)

model.fit(data)
forecast = model.predict(data)

In [11]:
forecast.head(3)

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,daily,daily_lower,daily_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2018-09-25 14:01:00,111.442675,132.559231,187.331496,111.442675,111.442675,48.903208,48.903208,48.903208,48.903208,48.903208,48.903208,0.0,0.0,0.0,160.345883
1,2018-09-25 14:02:00,111.444066,130.068874,191.579090,111.444066,111.444066,48.968552,48.968552,48.968552,48.968552,48.968552,48.968552,0.0,0.0,0.0,160.412618
2,2018-09-25 14:03:00,111.445456,135.303443,185.745744,111.445456,111.445456,49.030363,49.030363,49.030363,49.030363,49.030363,49.030363,0.0,0.0,0.0,160.475819


In [12]:
forecast[['ds', 'trend', 'yhat', 'yhat_lower', 'yhat_upper']].head(3)

,ds,trend,yhat,yhat_lower,yhat_upper
0,2018-09-25 14:01:00,111.442675,160.345883,132.559231,187.331496
1,2018-09-25 14:02:00,111.444066,160.412618,130.068874,191.579090
2,2018-09-25 14:03:00,111.445456,160.475819,135.303443,185.745744


In [13]:
forecast['real'] = data['y']
forecast['anomaly'] = 1
forecast.loc[forecast['real'] > forecast['yhat_upper'], 'anomaly'] = -1
forecast.loc[forecast['real'] < forecast['yhat_lower'], 'anomaly'] = -1
forecast.head(3)

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,daily,daily_lower,daily_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat,real,anomaly
0,2018-09-25 14:01:00,111.442675,132.559231,187.331496,111.442675,111.442675,48.903208,48.903208,48.903208,48.903208,48.903208,48.903208,0.0,0.0,0.0,160.345883,182.478,1
1,2018-09-25 14:02:00,111.444066,130.068874,191.579090,111.444066,111.444066,48.968552,48.968552,48.968552,48.968552,48.968552,48.968552,0.0,0.0,0.0,160.412618,176.231,1
2,2018-09-25 14:03:00,111.445456,135.303443,185.745744,111.445456,111.445456,49.030363,49.030363,49.030363,49.030363,49.030363,49.030363,0.0,0.0,0.0,160.475819,183.917,1


In [14]:
px.scatter(forecast, x='ds', y='real', color='anomaly', color_continuous_scale='Bluyl',
           title='Anomaly Detection in Time Series', template='plotly_dark', width=950, height=500)